The contents of this directory benchmark various model approaches. Results are saved as
1. Actual models
2. Slurm logs
3. HTML Dask performance reports

# Imports

In [30]:
# Imports
import subprocess
from datetime import datetime
import pickle
import os

import numpy as np
import pandas as pd

import statsmodels
from statsmodels.discrete.count_model import ZeroInflatedNegativeBinomialResultsWrapper as smzinb

# Submitting model runs

In [31]:
#functions for model submission...

data_root="/gpfs/gibbs/pi/reilly/tabula_data"

def bench(model_code):
    """
    Executes a particular model design & collects statistics. 
    t is time in hours
    """
    now=datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    
    logdir=f"{data_root}/speed_test/logs/{model_code}_{now}"

    #make a directory to put all the log files : this will be 
    os.makedirs(logdir, exist_ok=True)

    command=f"""
    module load miniconda
    conda activate biopython
    code_location=$(pwd)
    cd {logdir}
    python ${{code_location}}/cluster.py {model_code}
    """


    slurm_cmd = [
        "sbatch",
        "--partition=ycga",
        f"--time={t}:00:00",
        f"--output={logdir}/master_{model_code}_{now}.out",
        "-c 1",
        f"-J {model_code}_master",
        "--wrap", command
    ]
    result = subprocess.run(
        slurm_cmd, 
        capture_output=True, 
        text=True
    )
    
    print(result.stdout.strip() if result.returncode == 0 else result.stderr.strip())



In [39]:
bench("c9100090")

Submitted batch job 50744991


In [40]:
bench("c9100010")
bench("c9200090")

Submitted batch job 50745048
Submitted batch job 50745049


In [45]:
bench("c9200010")
bench("c0100000")
bench("c0100010")

Submitted batch job 50745107
Submitted batch job 50745108
Submitted batch job 50745109


In [46]:
bench("c0200000")

Submitted batch job 50745131


In [47]:
bench("c0200010")

Submitted batch job 50745136


# Summarizing success / failure
& everything in-between

In [33]:
def extract_parameters():
    """
    Takes a model & extracts triplets (pi, sigma-squared, mu)
    (break by type...)
    """
    pass

def load_model(model_file:str):
    """
    What is says on the tin. Points to relevant archive path to avoid retyping.
    """
    with open(f"{data_root}/speed_test/models/arch/{model_file}","rb") as f:
        return pickle.load(f)

def eval_single_statsmodels(model):
    """
    Takes a single statsmodels model & returns some useful QC metrics
    """
    cov=-1
    finite_cov=-1
    try:
        cov = model.cov_params()
        finite_cov = np.all(np.isfinite(cov.values))
    except (ValueError, np.linalg.LinAlgError) as e:
        finite_cov = False
        cov = None

    converged = model.mle_retvals.get("converged", False)
    finite_se = np.all(np.isfinite(model.bse)) if hasattr(model, "bse") else False

    return {
        "converged": converged,
        "covariance_matrix_available": cov is not None,
        "covariance_matrix_finite": finite_cov,
        "standard_errors_finite": finite_se
        #"cov_type": getattr(model, "cov_type", None),
    }
    return ret

def eval_model(model):
    """
    Produces QC metrics on a single model, regardless of type
    """
    if isinstance(model,smzinb):
        #print("[+] Unified statsmodels")
        return pd.DataFrame.from_dict(eval_single_statsmodels(model), orient='index', columns=['only'])
        
    elif isinstance(model,dict):
        all_zinb = [
            isinstance(model[key], smzinb)
            for key in model
        ]
        if all(all_zinb):
            #print("[+] Broken statsmodels")
            return pd.DataFrame({
                key:eval_single_statsmodels(model[key])
                for key in model
            })

        else:
            print("[!] Type not implemented yet. Aborting.")
    else:
        print("[!] Type not implemented yet. Aborting.")



def eval_models(modelpaths):
    """
    Loads and evaluates models, taking a list of filenames
    and returning a dictionary of model IDs pointing at
    evaluations
    """
    evals={}
    for path in modelpaths:
        name=path.split("_")[0]
        model=load_model(path)
        evals[name]=eval_model(model)
        del model
    return evals

def compare_models(evals):
    """
    Takes output of eval_models and compresses
    into a summary dataframe. 
    """
    summaries=[]
    for name in evals:
        summary=evals[name].sum(axis=1)
        summary["total"]=len(evals[name].columns)
        summary["model"]=name
        summaries.append(summary)
    summaries=pd.DataFrame(summaries)
    summaries.set_index(summaries["model"],inplace=True)
    summaries.drop("model",axis=1,inplace=True)
    return summaries


In [34]:
def eval_all():
    file_names=[f for f in os.listdir(f"{data_root}/speed_test/models/arch/")]
    evals=eval_models(file_names)
    return evals

def summarize_all():
    return compare_models(eval_all())



In [43]:
summary=summarize_all()

In [44]:
summary

,converged,covariance_matrix_available,covariance_matrix_finite,standard_errors_finite,total
model,,,,,
c9100010,2,2,2,2,2
c9100090,1,1,1,1,1
c9200090,1,1,1,1,1


Adding metadata so we don't have to memorize all the IDs.

In [26]:
modelspecs=pd.read_csv(f"{data_root}/speed_test/modelspecs.tsv",sep="\t")
modelspecs.set_index(modelspecs["code"],inplace=True)
modelspecs.drop("code",axis=1,inplace=True)

In [29]:
summary.join(modelspecs,how="inner")

,converged,covariance_matrix_available,covariance_matrix_finite,standard_errors_finite,total,dataset,lib,hardware,dna,main_equ_type,z_equ_type,main_equ,z_equ,broken_by
c000000,1,0,0,0,1,Shendure (0),statsmodels (0),CPU (0),No dna (0),simple addition (0),replicate (0),umis_mpra_bc ~ C(cre_id) + C(cell_type) -1,C(rep_id),NaN
c000010,10,6,6,6,10,Shendure (0),statsmodels (0),CPU (0),No dna (0),by cell-type (1),replicate (0),umis_mpra_bc ~ C(cre_id)-1,C(rep_id),cell_type
c900010,2,2,2,2,2,Fake (9),statsmodels (0),CPU (0),No dna (0),by cell-type (1),replicate (0),umi_count ~ C(cre_id)-1,C(rep_id),cell_type
c900090,1,1,1,1,1,Fake (9),statsmodels (0),CPU (0),No dna (0),simple interaction (9),replicate (0),umi_count ~ C(cre_id)*C(cell_type)-1,C(rep_id),NaN


# Summarizing performance

In [49]:
def format_seconds(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = seconds % 60
    return f"{hours}h {minutes}m {secs:.2f}s"

def simple_time(model_file):
    """
    Produces a quick time estimate from log files.
    Not as accurate or detailed as the full HTML report.
    """
    directory=f"{data_root}/speed_test/logs/arch/{model_file}"
    master_files = [f for f in os.listdir(directory) if f.startswith("master")]
    if len(master_files) != 1:
        raise ValueError(f"Expected exactly one 'master*' file, found {len(master_files)}.")
    
    filepath = os.path.join(directory, master_files[0])

    # dump file into memory (its small)
    with open(filepath, 'r') as file:
        lines = file.readlines()

    # Look for the line of interest and extract the last field
    for line in lines:
        if line.startswith("[+] Done with all tasks."):
            return format_seconds(float(line.strip().split()[-1]))
    
    raise ValueError("Could not find final timestamp. Did the model finish?")


simple_time("c9100010_2025-04-23_17-06-53")

'0h 0m 16.13s'

In [53]:
def simple_time_all():
    """
    returns a two column data-frame
    index is model code, `time` is time in seconds. 
    """
    directory=f"{data_root}/speed_test/logs/arch/"
    runs= [f for f in os.listdir(directory)]
    return {
        run_name.split("_")[0]:
        simple_time(run_name)
        for run_name in runs
    }

print(simple_time_all())

{'c9100010': '0h 0m 16.13s', 'c9100090': '0h 0m 14.59s', 'c9200090': '0h 0m 17.14s'}
